# B100 Stock Market Listed Companies
## Notebook 6: Revenue Trend Forecasting

This notebook uses a simple time-series regression and historical growth patterns to generate a 1-year revenue forecast for the Nifty 100 companies.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

clean_path = r"../data/clean"
profitloss = pd.read_csv(os.path.join(clean_path, "profitloss_clean.csv")).rename(columns={"company_id": "symbol", "year": "year_str"})

# Clean and convert year_str to integer year
import re
def extract_year(y_str):
    nums = re.findall(r'\d+', str(y_str))
    if nums:
        val = int(nums[0])
        return 2000 + val if val < 100 else val
    return np.nan

profitloss['year_int'] = profitloss['year_str'].apply(extract_year)
df_forecast = profitloss.dropna(subset=['year_int', 'sales'])
df_forecast = df_forecast[df_forecast['year_str'] != 'TTM']

forecasts = []
for symbol, group in df_forecast.groupby('symbol'):
    if len(group) >= 3:
        group = group.sort_values('year_int')
        X = group[['year_int']].values
        y = group['sales'].values
        
        model = LinearRegression()
        model.fit(X, y)
        
        next_year = int(group['year_int'].max() + 1)
        predicted_sales = max(0.0, float(model.predict([[next_year]])[0]))
        
        forecasts.append({
            "symbol": symbol,
            "current_year": int(group['year_int'].max()),
            "current_sales": float(y[-1]),
            "forecast_year": next_year,
            "forecasted_sales": round(predicted_sales, 2)
        })

forecast_df = pd.DataFrame(forecasts)
forecast_df.to_csv(r"../data/clean/forecasts_clean.csv", index=False)
print("1-Year Revenue forecasting completed. Sample forecasts:")
print(forecast_df.head(10))